# Audit the AI-written pipeline

A colleague asked a chat assistant for "a churn model in scikit-learn" and pasted the answer below. It runs and prints an accuracy of 0.82. You are the reviewer: find the four faults that make that number untrustworthy, and measure what each one hides.

One row is one telco customer; `Churn` is `Yes` when the customer left in the last month. Work in pairs: one person types, the other reads the code and asks "which rows did that line learn from?". Swap after step 3.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## Step 1 · Read the code before you run it

For every line in the next cell, write whether it **learns** something from the data (a mean, a list of columns, weights) and, if so, from which rows: all rows, or training rows only. Write your notes in the cell after it.

## Step 2 · Run it

Note the accuracy and the number of test customers.

In [ ]:
# The AI-written pipeline, as given
df = pd.read_csv("data/telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
y = (df["Churn"] == "Yes").astype(int)
X = pd.get_dummies(df.drop(columns=["customerID", "Churn"]))
X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
print("Accuracy:", accuracy_score(y_test, model.predict(X_test)))

In [ ]:
# Step 1 notes: which lines learn, and from which rows?
# Step 2: how many test customers? How many columns after get_dummies?

## Step 3 · Finding 1: did anything learn from the test customers?

Compare the mean of `tenure` over all rows with its mean over the training rows only. Then rewrite the code so the scaler sits inside a `make_pipeline` that is fitted on the training rows only. Does the accuracy move?

## Step 4 · Finding 2: is 0.82 good?

Score a `DummyClassifier(strategy="most_frequent")` on the same split. Then count how many of the test customers who really churned the model flagged: `((pred == 1) & (y_test == 1)).sum()` against `y_test.sum()`.

## Step 5 · Finding 3: who decided which customers count as "will churn"?

`predict()` flags a customer when the predicted probability is above 0.5. Print how many test customers are flagged at 0.5, then use `predict_proba(X_test)[:, 1] >= 0.3` and count again: how many flagged, how many of them really churned?

## Step 6 · Finding 4: would another split give the same number?

Run the corrected pipeline from step 3 with `random_state` 0, 1, 2, 3 and 4 and print the five accuracies.

Then score the course pipeline from D01 5.6 with 5-fold cross-validation on its training rows. The next cell builds it for you: `TotalCharges` converted, the `ColumnTransformer` `pre`, and the stratified 80 / 20 split as `Xc_train`, `yc_train`. Print the mean and the standard deviation.

In [ ]:
# The course pipeline from D01 5.6, with its stratified 80 / 20 split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder

course = pd.read_csv("data/telco_churn.csv")
course["TotalCharges"] = pd.to_numeric(course["TotalCharges"], errors="coerce")
yc = (course["Churn"] == "Yes").astype(int)
Xc = course.drop(columns=["customerID", "Churn"])
num_cols = list(Xc.select_dtypes(include="number").columns)
cat_cols = list(Xc.select_dtypes(include=["str", "object"]).columns)
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.20, stratify=yc, random_state=42)
pipe = make_pipeline(pre, LogisticRegression(max_iter=1000))

## Step 7 · The review comment

Four lines for your colleague, each naming the code line, what it hides, and the fix.

## Check yourself